# 🌌 Fink/LSST — Alert Sky Map

Interactive sky map of Fink/LSST transient alerts.

**What's fixed vs previous version:**
- HiPS background: robust 4-strategy download (FITS → JPG → direct HTTP × 2)
- Coordinate grid: solid, labelled lines on both axes in both projections
- Section 1b: HiPS diagnostic cell — run this first to see exactly what fails

**Column naming (LSST DPDD schema):**
- `r:` prefix → diaSource table field (≠ spectral band `r`)
- `f:` prefix → Fink-computed field
- `r:band` ∈ {`u`, `g`, `r`, `i`, `z`, `y`}

In [ ]:
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
#%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from rubinlsstskyalerts.fink_tools  import (
    load_catalog, catalog_summary,
    plot_skymap_rect, plot_skymap_mollweide, plot_skymap_combined,
    TAG_STYLES, DEFAULT_TAG_STYLE, RUBIN_DDF,
    DARK_BG, TEXT_COL, MUTED_COL, BORDER_COL, GRID_COL,
    GALPLANE_COL, GALBAND_COL, GALCENTER_COL, DDF_COL,
    galactic_plane_radec, galactic_latitude_radec, _split_segments,
    ra_deg_to_hms, _ra_to_moll, _dec_to_moll, _build_legend,
    draw_radec_grid,
    fetch_hips_image, overlay_hips_background,
)


print('✓ fink_skymap_lib imported')

In [ ]:
DATASET_DIR = Path('fink_dataset')
cat = load_catalog(DATASET_DIR)

print(f'Catalog: {len(cat)} unique diaObjects')
print(f'RA  : {cat["r:ra"].min():.3f}° – {cat["r:ra"].max():.3f}°')
print(f'Dec : {cat["r:dec"].min():.3f}° – {cat["r:dec"].max():.3f}°')
print()
print(catalog_summary(cat)[['label','n_objects','ra_min','ra_max',
                              'dec_min','dec_max','snn_mean']].to_string())

---
## Parameters

In [ ]:
RA_UNIT       = 'hms'    # 'hms' or 'deg'
RA_GRID_STEP  = 2.0      # degrees
DEC_GRID_STEP = 2.0      # degrees

SHOW_GAL_PLANE = True
SHOW_GAL_BAND  = True
GAL_BAND_B     = 15.0    # degrees
SHOW_DDF       = True

TAGS_TO_SHOW   = None    # None = all
SAVE_FIGURES   = False

print('✓ Parameters set')

---
## 1b · HiPS sky background diagnostic

Run this cell **before** enabling `SKY_BACKGROUND = True` anywhere.
It tests all 4 download strategies and shows exactly what works.

In [ ]:
# ── Diagnose HiPS availability ────────────────────────────────────────────────
print('=' * 60)
print('HiPS background diagnostic')
print('=' * 60)

# 1. Check astroquery installation
try:
    import astroquery
    print(f'astroquery version : {astroquery.__version__}')
    from astroquery.hips2fits import hips2fits
    import inspect
    sig = inspect.signature(hips2fits.query)
    print(f'hips2fits.query signature: {sig}')
except ImportError as e:
    print(f'✗ astroquery not found: {e}')
    print('  → pip install astroquery')

# 2. Check requests
try:
    import requests
    print(f'requests version   : {requests.__version__}')
except ImportError:
    print('✗ requests not found  →  pip install requests')

# 3. Check PIL
try:
    from PIL import Image
    import PIL
    print(f'Pillow version     : {PIL.__version__}')
except ImportError:
    print('✗ Pillow not found  →  pip install Pillow  (needed for JPG strategy)')

print()
print('Testing download for COSMOS field (RA=150.1°, Dec=+2.2°, FOV=5°)...')
print()

img = fetch_hips_image(
    ra_center  = 150.1,
    dec_center = 2.2,
    fov_deg    = 5.0,
    hips_survey= 'CDS/P/DSS2/color',
    width_px   = 256,
    height_px  = 256,
    verbose    = True,
)

print()
if img is not None:
    print(f'✓ HiPS image downloaded successfully')
    print(f'  shape={img.shape}  dtype={img.dtype}')
    print(f'  value range: [{img.min():.3f}, {img.max():.3f}]')

    # Quick preview
    fig, ax = plt.subplots(1, 1, figsize=(4, 4), facecolor=DARK_BG)
    ax.imshow(img, origin='upper')
    ax.set_title('HiPS test: COSMOS (DSS2)', color=TEXT_COL, fontsize=9)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('✗ HiPS download failed — sky background will not be available.')
    print()
    print('Possible causes:')
    print('  • No network access in this environment')
    print('  • CDS server temporarily unavailable')
    print('  • astroquery version incompatibility')
    print()
    print('Alternative: use DSS2 greyscale survey: "CDS/P/DSS2/red"')
    print('or unWISE:   "CDS/P/unWISE/color"')

---
## 2 · Combined view: Mollweide + Rectangular

In [ ]:
# Set SKY_BACKGROUND=True only after the diagnostic above confirms it works
SKY_BACKGROUND = False
HIPS_SURVEY    = 'CDS/P/DSS2/color'

fig = plot_skymap_combined(
    cat,
    ra_unit             = RA_UNIT,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_galactic_band  = SHOW_GAL_BAND,
    galactic_band_b     = GAL_BAND_B,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    sky_background      = SKY_BACKGROUND,
    hips_survey         = HIPS_SURVEY,
    ra_grid_step        = RA_GRID_STEP,
    dec_grid_step       = DEC_GRID_STEP,
    tags_to_show        = TAGS_TO_SHOW,
    figsize             = (16, 12),
    save_path           = DATASET_DIR / 'skymap_combined.png' if SAVE_FIGURES else None,
)
plt.show()

---
## 3 · Mollweide full-sky only

In [ ]:
fig, ax = plot_skymap_mollweide(
    cat,
    marker_size         = 35,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_galactic_band  = SHOW_GAL_BAND,
    galactic_band_b     = GAL_BAND_B,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    ra_unit             = RA_UNIT,
    tags_to_show        = TAGS_TO_SHOW,
    figsize             = (14, 7),
)
if SAVE_FIGURES:
    fig.savefig(DATASET_DIR / 'skymap_mollweide.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 4 · Rectangular zoom — HMS labels

In [ ]:
fig, ax = plot_skymap_rect(
    cat,
    marker_size         = 50,
    marker_alpha        = 0.9,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_galactic_band  = SHOW_GAL_BAND,
    galactic_band_b     = GAL_BAND_B,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    ra_unit             = 'hms',
    ra_grid_step        = RA_GRID_STEP,
    dec_grid_step       = DEC_GRID_STEP,
    sky_background      = False,    # ← set True after diagnostic passes
    hips_survey         = HIPS_SURVEY,
    tags_to_show        = TAGS_TO_SHOW,
    figsize             = (14, 8),
)
if SAVE_FIGURES:
    fig.savefig(DATASET_DIR / 'skymap_rect_hms.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 5 · Rectangular — RA in degrees

In [ ]:
fig, ax = plot_skymap_rect(
    cat,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    ra_unit             = 'deg',
    ra_grid_step        = 5.0,
    dec_grid_step       = 5.0,
    tags_to_show        = TAGS_TO_SHOW,
    title               = 'Fink/LSST alerts — RA in degrees',
    figsize             = (14, 8),
)
plt.show()

--- 
## 6 · Mollweide with HiPS sky background

Only run this cell after the diagnostic (Section 1b) confirms the download works.
HiPS surveys must be rendered in **Mollweide projection** (equal-area full-sky).

In [ ]:
# Available HiPS surveys (examples):
#   'CDS/P/DSS2/color'           Digitized Sky Survey (optical colour)
#   'CDS/P/DSS2/red'             DSS2 red band (greyscale, faster)
#   'CDS/P/2MASS/color'          2MASS JHK near-infrared
#   'CDS/P/PanSTARRS/DR1/color'  PanSTARRS optical
#   'CDS/P/unWISE/color'         unWISE W1+W2 mid-infrared
#   'CDS/P/SDSS9/color'          SDSS optical

HIPS_SURVEY_CHOICE = 'CDS/P/DSS2/color'

fig, ax = plot_skymap_mollweide(
    cat,
    marker_size         = 55,
    marker_alpha        = 0.95,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_galactic_band  = SHOW_GAL_BAND,
    galactic_band_b     = GAL_BAND_B,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    ra_unit             = RA_UNIT,
    sky_background      = True,          # ← enabled
    hips_survey         = HIPS_SURVEY_CHOICE,
    hips_alpha          = 0.60,
    hips_verbose        = True,
    tags_to_show        = TAGS_TO_SHOW,
    title               = f'Fink/LSST alerts on {HIPS_SURVEY_CHOICE}',
    figsize             = (14, 7),
)
if SAVE_FIGURES:
    fig.savefig(DATASET_DIR / 'skymap_hips_mollweide.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 7 · Per-tag individual maps

In [ ]:
tags_in_data = sorted(cat['fink_tag'].unique())
MARGIN = 3.0   # degrees of padding

for tag in tags_in_data:
    sub = cat[cat['fink_tag'] == tag]

    fig, ax = plot_skymap_rect(
        cat,
        ra_min              = sub['r:ra'].min()  - MARGIN,
        ra_max              = sub['r:ra'].max()  + MARGIN,
        dec_min             = sub['r:dec'].min() - MARGIN,
        dec_max             = sub['r:dec'].max() + MARGIN,
        show_galactic_plane = SHOW_GAL_PLANE,
        show_galactic_band  = SHOW_GAL_BAND,
        galactic_band_b     = GAL_BAND_B,
        show_ddf            = SHOW_DDF,
        show_grid           = True,
        ra_unit             = RA_UNIT,
        ra_grid_step        = 2.0,
        dec_grid_step       = 2.0,
        tags_to_show        = [tag],
        title               = f'Tag: {tag}  ({len(sub)} objects)',
        figsize             = (12, 6),
    )
    if SAVE_FIGURES:
        fig.savefig(DATASET_DIR / f'skymap_{tag}.png',
                    dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()
    plt.close(fig)

---
## 8 · SNR-weighted marker size

In [ ]:
SNR_SCALE = 0.3

fig, ax = plot_skymap_rect(
    cat,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    ra_unit             = RA_UNIT,
    ra_grid_step        = RA_GRID_STEP,
    dec_grid_step       = DEC_GRID_STEP,
    tags_to_show        = [],    # suppress default scatter
    title               = 'Fink/LSST — marker size ∝ SNR',
    figsize             = (14, 8),
)

for tag in sorted(cat['fink_tag'].unique()):
    s    = TAG_STYLES.get(tag, DEFAULT_TAG_STYLE)
    sub  = cat[cat['fink_tag'] == tag]
    sizes = np.clip(sub['r:snr'].fillna(5).values * SNR_SCALE, 8, 350)
    ax.scatter(sub['r:ra'], sub['r:dec'],
               c=s['color'], marker=s['marker'],
               s=sizes, alpha=0.85, zorder=s['zorder'],
               edgecolors='white', linewidths=0.3,
               label=f"{s['label']}  (n={len(sub)})")

# Size legend
for snr_ref in [10, 50, 200]:
    ax.scatter([], [], c='gray', marker='o', s=snr_ref*SNR_SCALE,
               label=f'SNR={snr_ref}', edgecolors='white', linewidths=0.3)

_build_legend(ax, SHOW_DDF, SHOW_GAL_PLANE, SHOW_GAL_BAND, GAL_BAND_B)
plt.show()

---
## 9 · SNN score as continuous colormap

In [ ]:
import matplotlib.colors as mcolors

fig, ax = plot_skymap_rect(
    cat,
    show_galactic_plane = SHOW_GAL_PLANE,
    show_ddf            = SHOW_DDF,
    show_grid           = True,
    ra_unit             = RA_UNIT,
    ra_grid_step        = RA_GRID_STEP,
    dec_grid_step       = DEC_GRID_STEP,
    tags_to_show        = [],
    title               = 'Fink/LSST — SNN score (SN vs others)',
    figsize             = (14, 8),
)

sc = ax.scatter(
    cat['r:ra'], cat['r:dec'],
    c    = cat['f:clf_snnSnVsOthers_score'].fillna(0),
    cmap = 'RdYlBu_r',
    vmin = 0, vmax = 1,
    s=55, alpha=0.9, zorder=5,
    edgecolors='white', linewidths=0.3,
)
cb = plt.colorbar(sc, ax=ax, fraction=0.025, pad=0.01)
cb.set_label('SNN score (SN vs others)', color=TEXT_COL, fontsize=9)
cb.ax.tick_params(labelsize=7, colors=TEXT_COL)

_build_legend(ax, SHOW_DDF, SHOW_GAL_PLANE, SHOW_GAL_BAND, GAL_BAND_B)

if SAVE_FIGURES:
    fig.savefig(DATASET_DIR / 'skymap_snn.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 10 · DDFs — cross-match and zoom

In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as _u

print(f'{"Name":12s}  {"RA (deg)":>10s}  {"RA (HMS)":>10s}  {"Dec (deg)":>10s}')
print('-' * 52)
for ddf in RUBIN_DDF:
    print(f"{ddf['name']:12s}  {ddf['ra']:10.4f}°  "
          f"{ra_deg_to_hms(ddf['ra']):>10s}  {ddf['dec']:+10.4f}°")

print()
MATCH_R = 3.0  # degrees
alert_coords = SkyCoord(ra=cat['r:ra'].values*_u.deg,
                        dec=cat['r:dec'].values*_u.deg)
for ddf in RUBIN_DDF:
    sep  = SkyCoord(ra=ddf['ra']*_u.deg, dec=ddf['dec']*_u.deg).separation(alert_coords).deg
    near = cat[sep < MATCH_R]
    print(f"{ddf['name']:12s}: {len(near)} alerts within {MATCH_R}°  "
          f"{near['fink_tag'].value_counts().to_dict() if len(near) else ''}")

In [ ]:
# Zoom on one DDF
DDF_NAME = 'COSMOS'
ZOOM_DEG = 5.0

ddf = next(d for d in RUBIN_DDF if d['name'] == DDF_NAME)

fig, ax = plot_skymap_rect(
    cat,
    ra_min  = ddf['ra']  - ZOOM_DEG,
    ra_max  = ddf['ra']  + ZOOM_DEG,
    dec_min = ddf['dec'] - ZOOM_DEG,
    dec_max = ddf['dec'] + ZOOM_DEG,
    show_galactic_plane = True,
    show_ddf            = True,
    show_grid           = True,
    ra_unit             = RA_UNIT,
    ra_grid_step        = 1.0,
    dec_grid_step       = 1.0,
    tags_to_show        = TAGS_TO_SHOW,
    title               = f'Zoom on {DDF_NAME}  (±{ZOOM_DEG}°)',
    figsize             = (10, 8),
)

from matplotlib.patches import Circle
ax.add_patch(Circle((ddf['ra'], ddf['dec']), ZOOM_DEG * 0.8,
                    fill=False, edgecolor=DDF_COL, lw=1.3,
                    ls='--', transform=ax.transData, zorder=12))
plt.show()

---
## 11 · 2D alert density histogram

In [ ]:
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(14, 7), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)

h, xedge, yedge, img = ax.hist2d(
    cat['r:ra'], cat['r:dec'],
    bins=40, cmap='YlOrRd',
    norm=mcolors.LogNorm(),
    zorder=2,
)
cb = plt.colorbar(img, ax=ax, fraction=0.025, pad=0.01)
cb.set_label('N alerts (log)', color=TEXT_COL, fontsize=9)
cb.ax.tick_params(labelsize=7, colors=TEXT_COL)

# Galactic plane
ra_min_d, ra_max_d   = cat['r:ra'].min()-2,  cat['r:ra'].max()+2
dec_min_d, dec_max_d = cat['r:dec'].min()-2, cat['r:dec'].max()+2
_draw_galactic_rect = lambda: [  # inline
    ax.plot(ra_s[m], dec_s[m], color=GALPLANE_COL, lw=1.8, alpha=0.9, zorder=5)
    for ra_s, dec_s in _split_segments(*galactic_plane_radec(3000))
    for m in [((ra_s>=ra_min_d)&(ra_s<=ra_max_d)&
               (dec_s>=dec_min_d)&(dec_s<=dec_max_d))]
    if m.sum() > 1
]
_draw_galactic_rect()

# Grid
draw_radec_grid(ax, ra_min_d, ra_max_d, dec_min_d, dec_max_d,
                ra_step=5.0, dec_step=5.0, ra_unit=RA_UNIT)

ax.invert_xaxis()
ax.set_xlabel("Right Ascension", color=TEXT_COL, fontsize=10)
ax.set_ylabel("Declination (degrees)", color=TEXT_COL, fontsize=10)
ax.set_title('Alert density map — Fink/LSST',
             color=TEXT_COL, fontsize=11, fontfamily='monospace')
for sp in ax.spines.values(): sp.set_edgecolor(BORDER_COL)

if SAVE_FIGURES:
    fig.savefig(DATASET_DIR / 'skymap_density.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()